# Trigger Ingestion Pipeline - 创建新向量数据库

这个 Notebook 提供三种方式来触发 ingestion pipeline 创建新的向量数据库：

1. **方式一**: 从 URL 下载 PDF（最简单）
2. **方式二**: 从 MinIO/S3 读取文档
3. **方式三**: 从 GitHub 仓库读取文档

## 前提条件
- 已连接到 OpenShift 集群
- RAG 项目已部署在 `llama-stack-rag` namespace

## 安装依赖

In [ ]:
!pip install requests

## 配置环境变量

你需要先获取 `rag-ingestion-pipeline` 服务的访问方式。

### 选项 A: 通过 Port-Forward（推荐用于本地测试）
在终端运行：
```bash
oc port-forward -n llama-stack-rag svc/rag-ingestion-pipeline 8000:80
```
然后使用 `http://localhost:8000`

### 选项 B: 从集群内部访问（如在 Jupyter Notebook Pod 中）
直接使用 `http://rag-ingestion-pipeline` 或 `http://rag-ingestion-pipeline.llama-stack-rag.svc.cluster.local`

In [ ]:
import requests
import json
from datetime import datetime

# ============================================
# 配置 - 根据你的环境修改
# ============================================

# 如果使用 port-forward，使用 localhost
# INGESTION_PIPELINE_URL = "http://localhost:8000"

# 如果从集群内部访问（如 Jupyter Notebook Pod）
INGESTION_PIPELINE_URL = "http://rag-ingestion-pipeline"

print(f"Ingestion Pipeline URL: {INGESTION_PIPELINE_URL}")

## 验证服务可用性

In [ ]:
def check_service_health():
    """检查 ingestion-pipeline 服务是否可用"""
    try:
        response = requests.get(f"{INGESTION_PIPELINE_URL}/health", timeout=10)
        if response.status_code == 200:
            print("✅ Ingestion Pipeline 服务正常运行")
            return True
        else:
            print(f"⚠️ 服务返回状态码: {response.status_code}")
            return False
    except requests.exceptions.ConnectionError:
        print("❌ 无法连接到 Ingestion Pipeline 服务")
        print("请确保:")
        print("  1. 你在集群内部运行此 notebook，或")
        print("  2. 已运行 port-forward: oc port-forward -n llama-stack-rag svc/rag-ingestion-pipeline 8000:80")
        return False

check_service_health()

---
## 方式一: 从 URL 创建向量数据库（最简单）

直接从公开的 PDF URL 下载文档并创建向量数据库。

In [ ]:
def create_vectordb_from_urls(
    name: str,
    version: str,
    urls: list,
    embedding_model: str = "qwen-3-4b-embedding/qwen3-4b-embedding"
):
    """
    从 URL 列表创建向量数据库
    
    Args:
        name: 向量数据库名称
        version: 版本号（如 "1.0"）
        urls: PDF 文档 URL 列表
        embedding_model: 嵌入模型名称
    """
    vector_store_name = f"{name}-v{version.replace('.', '-')}"
    
    payload = {
        "name": name,
        "version": version,
        "source": "URL",
        "embedding_model": embedding_model,
        "vector_store_name": vector_store_name,
        "urls": urls
    }
    
    print(f"📤 提交请求到 {INGESTION_PIPELINE_URL}/add")
    print(f"📝 Payload: {json.dumps(payload, indent=2)}")
    
    response = requests.post(
        f"{INGESTION_PIPELINE_URL}/add",
        json=payload,
        headers={"Content-Type": "application/json"}
    )
    
    if response.status_code == 200:
        result = response.json()
        print(f"\n✅ Pipeline 创建成功!")
        print(f"📊 响应: {json.dumps(result, indent=2)}")
        return result
    else:
        print(f"\n❌ 请求失败: {response.status_code}")
        print(f"错误信息: {response.text}")
        return None

In [ ]:
# 示例: 从 arXiv PDF 创建向量数据库

result = create_vectordb_from_urls(
    name="docling-paper-db",
    version="1.0",
    urls=[
        "https://arxiv.org/pdf/2408.09869",  # Docling 论文
    ]
)

---
## 方式二: 从 MinIO/S3 创建向量数据库

从 MinIO 或 S3 bucket 读取文档。需要先上传文档到 bucket。

In [ ]:
def create_vectordb_from_s3(
    name: str,
    version: str,
    bucket_name: str,
    endpoint_url: str = "http://minio:9000",
    access_key_id: str = "minio_rag_user",
    secret_access_key: str = "minio_rag_password",
    region: str = "us-east-1",
    embedding_model: str = "qwen-3-4b-embedding/qwen3-4b-embedding"
):
    """
    从 S3/MinIO bucket 创建向量数据库
    
    Args:
        name: 向量数据库名称
        version: 版本号
        bucket_name: S3 bucket 名称
        endpoint_url: MinIO/S3 endpoint
        access_key_id: 访问密钥 ID
        secret_access_key: 访问密钥
        region: AWS 区域
        embedding_model: 嵌入模型名称
    """
    vector_store_name = f"{name}-v{version.replace('.', '-')}"
    
    payload = {
        "name": name,
        "version": version,
        "source": "S3",
        "embedding_model": embedding_model,
        "vector_store_name": vector_store_name,
        "access_key_id": access_key_id,
        "secret_access_key": secret_access_key,
        "endpoint_url": endpoint_url,
        "bucket_name": bucket_name,
        "region": region
    }
    
    print(f"📤 提交请求到 {INGESTION_PIPELINE_URL}/add")
    print(f"📝 Payload (隐藏密钥): {json.dumps({**payload, 'secret_access_key': '***'}, indent=2)}")
    
    response = requests.post(
        f"{INGESTION_PIPELINE_URL}/add",
        json=payload,
        headers={"Content-Type": "application/json"}
    )
    
    if response.status_code == 200:
        result = response.json()
        print(f"\n✅ Pipeline 创建成功!")
        print(f"📊 响应: {json.dumps(result, indent=2)}")
        return result
    else:
        print(f"\n❌ 请求失败: {response.status_code}")
        print(f"错误信息: {response.text}")
        return None

In [ ]:
# 示例: 从 MinIO 的 documents bucket 创建向量数据库
# 注意: 需要先上传 PDF 到 MinIO

result = create_vectordb_from_s3(
    name="my-minio-vectordb",
    version="1.0",
    bucket_name="documents",
    endpoint_url="http://minio:9000",
    access_key_id="minio_rag_user",
    secret_access_key="minio_rag_password"
)

---
## 方式三: 从 GitHub 仓库创建向量数据库

从 GitHub 仓库的指定路径读取 PDF 文档。

In [ ]:
def create_vectordb_from_github(
    name: str,
    version: str,
    repo_url: str,
    path: str,
    branch: str = "main",
    token: str = "",
    embedding_model: str = "qwen-3-4b-embedding/qwen3-4b-embedding"
):
    """
    从 GitHub 仓库创建向量数据库
    
    Args:
        name: 向量数据库名称
        version: 版本号
        repo_url: GitHub 仓库 URL
        path: 仓库内的文档路径
        branch: 分支名称
        token: GitHub token (可选，用于私有仓库)
        embedding_model: 嵌入模型名称
    """
    vector_store_name = f"{name}-v{version.replace('.', '-')}"
    
    payload = {
        "name": name,
        "version": version,
        "source": "GITHUB",
        "embedding_model": embedding_model,
        "vector_store_name": vector_store_name,
        "url": repo_url,
        "path": path,
        "branch": branch,
        "token": token
    }
    
    print(f"📤 提交请求到 {INGESTION_PIPELINE_URL}/add")
    display_payload = {**payload}
    if token:
        display_payload['token'] = '***'
    print(f"📝 Payload: {json.dumps(display_payload, indent=2)}")
    
    response = requests.post(
        f"{INGESTION_PIPELINE_URL}/add",
        json=payload,
        headers={"Content-Type": "application/json"}
    )
    
    if response.status_code == 200:
        result = response.json()
        print(f"\n✅ Pipeline 创建成功!")
        print(f"📊 响应: {json.dumps(result, indent=2)}")
        return result
    else:
        print(f"\n❌ 请求失败: {response.status_code}")
        print(f"错误信息: {response.text}")
        return None

In [ ]:
# 示例: 从 RAG 项目的 notebooks/hr 目录创建向量数据库

result = create_vectordb_from_github(
    name="github-hr-vectordb",
    version="1.0",
    repo_url="https://github.com/rh-ai-quickstart/RAG.git",
    path="notebooks/hr",
    branch="main"
)

---
## 验证结果

Pipeline 提交后，需要等待几分钟让它完成。然后可以验证向量数据库是否创建成功。

In [ ]:
def list_vector_stores(llamastack_url: str = "http://llamastack:8321"):
    """列出所有向量数据库"""
    try:
        response = requests.get(f"{llamastack_url}/v1/vector_stores", timeout=10)
        if response.status_code == 200:
            data = response.json()
            print(f"📊 找到 {len(data.get('data', []))} 个向量数据库:\n")
            for vs in data.get('data', []):
                print(f"  • {vs['name']}")
                print(f"    ID: {vs['id']}")
                print(f"    状态: {vs['status']}")
                print(f"    文件数: {vs['file_counts']['total']}")
                print()
            return data
        else:
            print(f"❌ 请求失败: {response.status_code}")
            return None
    except requests.exceptions.ConnectionError:
        print("❌ 无法连接到 LlamaStack 服务")
        return None

# 列出所有向量数据库
list_vector_stores()

In [ ]:
def check_pipeline_status():
    """检查最近的 Pipeline 运行状态 (需要 oc 命令)"""
    import subprocess
    try:
        result = subprocess.run(
            ["oc", "get", "workflow", "-n", "llama-stack-rag", "--sort-by=.metadata.creationTimestamp"],
            capture_output=True,
            text=True
        )
        print("📊 最近的 Pipeline 运行状态:\n")
        print(result.stdout)
        if result.stderr:
            print(f"⚠️ {result.stderr}")
    except FileNotFoundError:
        print("❌ oc 命令不可用。请在终端运行:")
        print("   oc get workflow -n llama-stack-rag")

check_pipeline_status()

---
## 测试 RAG 查询

使用新创建的向量数据库进行 RAG 查询。

In [ ]:
def test_rag_query(
    query: str,
    vector_store_name: str,
    llamastack_url: str = "http://llamastack:8321"
):
    """
    测试 RAG 查询
    
    Args:
        query: 查询问题
        vector_store_name: 向量数据库名称
        llamastack_url: LlamaStack URL
    """
    # 首先进行向量搜索
    search_payload = {
        "query": query,
        "max_chunks": 5
    }
    
    print(f"🔍 查询: {query}")
    print(f"📚 向量数据库: {vector_store_name}")
    print("-" * 50)
    
    # 使用 rag_tool 进行查询
    try:
        response = requests.post(
            f"{llamastack_url}/v1/tool-runtime/rag-tool/query",
            json={
                "content": query,
                "vector_db_ids": [vector_store_name]
            },
            timeout=30
        )
        
        if response.status_code == 200:
            result = response.json()
            print("\n📄 检索到的相关内容:")
            for i, chunk in enumerate(result.get('chunks', [])[:3], 1):
                content = chunk.get('content', '')[:200]
                print(f"\n--- Chunk {i} ---")
                print(content + "..." if len(chunk.get('content', '')) > 200 else content)
            return result
        else:
            print(f"❌ 查询失败: {response.status_code}")
            print(response.text)
            return None
    except Exception as e:
        print(f"❌ 错误: {e}")
        return None

In [ ]:
# 示例: 测试 RAG 查询
# 请将 vector_store_name 替换为你创建的向量数据库名称

test_rag_query(
    query="What is Docling and how does it work?",
    vector_store_name="docling-paper-db-v1-0"  # 替换为你的向量数据库名称
)

---
## 快速参考: API Schema

### URLsSource
```json
{
    "name": "string",
    "version": "string",
    "source": "URL",
    "embedding_model": "qwen-3-4b-embedding/qwen3-4b-embedding",
    "vector_store_name": "string",
    "urls": ["https://example.com/doc.pdf"]
}
```

### S3Source
```json
{
    "name": "string",
    "version": "string",
    "source": "S3",
    "embedding_model": "qwen-3-4b-embedding/qwen3-4b-embedding",
    "vector_store_name": "string",
    "access_key_id": "string",
    "secret_access_key": "string",
    "endpoint_url": "http://minio:9000",
    "bucket_name": "string",
    "region": "us-east-1"
}
```

### GitHubSource
```json
{
    "name": "string",
    "version": "string",
    "source": "GITHUB",
    "embedding_model": "qwen-3-4b-embedding/qwen3-4b-embedding",
    "vector_store_name": "string",
    "url": "https://github.com/org/repo.git",
    "path": "docs",
    "branch": "main",
    "token": ""  // optional
}
```